# Transcripción del conjunto intermedio

Siete movimientos de nivel intermedio con edición Braille compás sobre compás de la biblioteca BrailleOrch y grabación libre. Sirven para ver cómo se comporta el sistema en repertorio más sencillo que las Variaciones Goldberg, con textura de melodía y acompañamiento y pocas voces. No se usan para elegir parámetros.

| Obra | Braille | Grabación | Licencia | Formato |
| --- | --- | --- | --- | --- |
| Beethoven, Minueto en Sol WoO 10 n.º 2 | Bor087 | Peter Johnston, Musopen, en Wikimedia Commons | dominio público | MP3 |
| Clementi, Sonatina Op. 36 n.º 1, tres movimientos | Bor360 | Alessio Cesaroni, Wikimedia Commons | CC0 | MP3 |
| Beethoven, Sonatina en Sol Anh. 5, Moderato y Romanza | Bor361 | StasNick, Jamendo en Internet Archive | CC BY-SA 3.0 | MP3 |
| MacDowell, To a Wild Rose Op. 51 n.º 1 | Bor194 | Gregory Tait, Internet Archive | CC BY 4.0 | FLAC |

Wikimedia limita las descargas desde Kaggle, así que las mismas grabaciones se adjuntan como el dataset `amt-intermedio-audio`; sin él, se descargan de sus fuentes. El notebook solo guarda las notas, igual que `amt-externo-transcribe`.

## 1. Instalación

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("--no-deps", "piano_transcription_inference", "torchlibrosa", "mido")
pip("librosa", "soundfile")

## 2. Grabaciones

In [ ]:
import glob, json, time, urllib.parse, urllib.request
from pathlib import Path

AUDIO = Path("/kaggle/working/audio")
AUDIO.mkdir(parents=True, exist_ok=True)
COMMONS = "https://upload.wikimedia.org/wikipedia/commons/transcoded/"
OBRAS = {
    "minueto_sol": COMMONS + "d/db/Minuet_in_G_%28Beethoven%29%2C_piano.ogg/Minuet_in_G_%28Beethoven%29%2C_piano.ogg.mp3",
    "clementi_1": COMMONS + "e/e7/Clementi%2C_Sonatina_N.1_-_1_Mov._Allegro.ogg/Clementi%2C_Sonatina_N.1_-_1_Mov._Allegro.ogg.mp3",
    "clementi_2": COMMONS + "b/b3/Clementi%2C_Sonatina_N.1_-_2_Mov._Andante.ogg/Clementi%2C_Sonatina_N.1_-_2_Mov._Andante.ogg.mp3",
    "clementi_3": COMMONS + "5/5c/Clementi%2C_Sonatina_N.1_-_3_Mov._Vivace.ogg/Clementi%2C_Sonatina_N.1_-_3_Mov._Vivace.ogg.mp3",
    "sonatina_sol_1": "https://archive.org/download/jamendo-317079/" + urllib.parse.quote("01-1570994-StasNick-Beethoven - Sonatina in G major, Anh. 5, Moderato _18-08-2018_.mp3"),
    "sonatina_sol_2": "https://archive.org/download/jamendo-314709/" + urllib.parse.quote("01-1570993-StasNick-Beethoven - Sonatina in G major, Anh. 5, Romanze _18-08-2018_.mp3"),
    "wild_rose": "https://archive.org/download/macdowell-woodland-sketches/" + urllib.parse.quote("MacDowell Woodland Sketch 1.flac"),
}
# Wikimedia limita las descargas desde Kaggle; si el dataset con las mismas grabaciones esta adjunto se copia de ahi
COPIA = next(iter(glob.glob("/kaggle/input/**/wild_rose.flac", recursive=True)), None)
for nombre, url in OBRAS.items():
    destino = AUDIO / (nombre + Path(urllib.parse.unquote(url)).suffix)
    if not destino.exists():
        if COPIA:
            destino.write_bytes((Path(COPIA).parent / destino.name).read_bytes())
        else:
            req = urllib.request.Request(url, headers={"User-Agent": "amt-braille-research/1.0"})
            destino.write_bytes(urllib.request.urlopen(req).read())
            time.sleep(4)
    print(nombre, destino.suffix, destino.stat().st_size // 1024, "KB")

## 3. Modelos y normalización

El normalizador es el mismo de `src/amt/audio.py`: quita la componente continua y lleva el RMS de los tramos con sonido a -24.9 dBFS sin recortar picos.

In [ ]:
import glob
import numpy as np
import librosa
import torch
from piano_transcription_inference import PianoTranscription, sample_rate as SR

TARGET_DBFS = -24.9

def db(x):
    return 20.0 * np.log10(max(x, 1e-12))

def active_rms(x, frame=2048):
    n = len(x) // frame
    rms = np.sqrt(np.mean(x[: n * frame].reshape(n, frame) ** 2, axis=1))
    loud = rms[rms >= rms.max() * 10 ** (-40 / 20)]
    return float(np.sqrt(np.mean(loud ** 2)))

def normalize_loudness(x):
    x = np.asarray(x, dtype=np.float32) - np.mean(x)
    level = active_rms(x)
    if db(level) < -80:
        return x
    gain = 10 ** ((TARGET_DBFS - db(level)) / 20)
    gain = min(gain, 0.99 / (np.abs(x).max() + 1e-12))
    return (x * gain).astype(np.float32)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE = str(Path.home() / "piano_transcription_inference_data" / "note_F1=0.9677_pedal_F1=0.9186.pth")
TUNED = glob.glob("/kaggle/input/**/kong_ajustado.pth", recursive=True)
assert TUNED, "Agregar la salida de amt-finetune como input del notebook"
MODELS = {"original": BASE, "ajustado": TUNED[0]}
print(DEVICE, MODELS)

## 4. Transcripción

In [ ]:
OUT = Path("/kaggle/working/intermedio")
OUT.mkdir(exist_ok=True)

for model_name, ckpt in MODELS.items():
    transcriber = PianoTranscription(device=DEVICE, checkpoint_path=ckpt)
    for nombre in OBRAS:
        destino = OUT / f"{model_name}__{nombre}.json"
        if destino.exists():
            continue
        audio, _ = librosa.load(str(next(AUDIO.glob(nombre + ".*"))), sr=SR, mono=True)
        audio = normalize_loudness(audio)
        t0 = time.time()
        out = transcriber.transcribe(audio, None)
        latency = time.time() - t0
        notes = [[float(e["onset_time"]), float(e["offset_time"]), int(e["midi_note"]), int(e["velocity"])]
                 for e in out["est_note_events"]]
        pedals = [[float(e["onset_time"]), float(e["offset_time"])] for e in out.get("est_pedal_events", [])]
        destino.write_text(json.dumps({
            "pieza": nombre, "archivo": urllib.parse.unquote(OBRAS[nombre].rsplit("/", 1)[1]), "modelo": model_name,
            "duracion_s": len(audio) / SR, "latencia_s": latency,
            "notas": notes, "pedal": pedals,
        }))
        print(f"{model_name:9s} {nombre:12s} {len(notes):5d} notas  {latency:5.1f} s")
    del transcriber
    torch.cuda.empty_cache()